## **Alumno:** <span style="color:red">**Juan Manuel Resquin**</span>
# <span style="color:red">**Entrenamiento de Modelos**</span>
### En este proceso, ya comenzare a subir los resultados del entrenamiento a MLflow, conectandolo no solo con GitHub, sino con DagsHub.v

### <span style="color:red">Este notebook usa churn_env para el registro oficial en MLflow y exportación del modelo .</span>

In [3]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import dagshub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, accuracy_score, precision_score, f1_score, cohen_kappa_score, classification_report

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Conexión con DagsHub y MLflow 
import dagshub
dagshub.init(repo_owner='JuanManuelResquin84', repo_name='Proyecto_AndesLink', mlflow=True)

# Volvemos a cargar el archivo, que ya sabemos que esta todo en orden 
df = pd.read_csv('../data/churn_sintetico.csv')  
print(df.head())

Accessing as JuanManuelResquin84

Initialized MLflow to track repo "JuanManuelResquin84/Proyecto_AndesLink"

Repository JuanManuelResquin84/Proyecto_AndesLink initialized!

   tenure_months  monthly_charge  total_charges  support_tickets  \
0              7           58.23         326.50                2   
1             56           56.75        3154.21                0   
2             48           78.84        3864.31                3   
3             32           79.74        2511.40                0   
4             32           55.37        1735.51                3   

   late_payments  avg_monthly_usage_gb contract_type payment_method  \
0              1                 81.83       mensual  transferencia   
1              2                 96.52         anual         debito   
2              2                 93.60       bianual       efectivo   
3              0                 28.95       bianual         debito   
4              0                126.90         anual       efectivo   

  internet_service  has_streaming  has_security_pack  num_products  region  \
0            cable              0                  1             3  centro   
1       

# **Preparación de los datos para el entrenamiento**

In [4]:
df_ml = df.copy()

# Encoding para convertir los textos a números
# pd.get_dummies transformará automáticamente 'anual', 'mensual', etc., en columnas de 0 y 1
X = pd.get_dummies(df_ml.drop('churn', axis=1), drop_first=True)

# Variable Objetivo
# Usamos LabelEncoder para convertir 'Si/No' en 1/0.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df_ml['churn'])

# División del Dataset (80% para entrenamiento y 20% para test) y uso stratify para mantener la proporción de Churn en ambos sets
X_train_ml, X_test_ml, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# **Escalado**

In [5]:
# Creamos las variables X_train y X_test definitivas y lo escalamos
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_ml)
X_test = scaler.transform(X_test_ml)

# **Entrenamiento con el Modelo de Regrecion Logistica**

In [6]:
from sklearn.linear_model import LogisticRegression

# Registro el experimento en MLflow
with mlflow.start_run(run_name="Logistic_Regression_AndesLink"):
    
    # Configuro el modelo (con el max_iter me ayudara a que converja con datos escalados)
    model_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
   
    mlflow.log_param("model_type", "Logistic Regression")
    
    # Entrenamiento directo con mis variables
    model_lr.fit(X_train, y_train)
    
    # Predicciones
    y_pred_lr = model_lr.predict(X_test)
    
    # Cálculo de todas las métricas 
    acc = accuracy_score(y_test, y_pred_lr)
    prec = precision_score(y_test, y_pred_lr)
    rec = recall_score(y_test, y_pred_lr)
    f1 = f1_score(y_test, y_pred_lr)
    kap = cohen_kappa_score(y_test, y_pred_lr)
    
    # Registro en MLflow/DagsHub
    mlflow.log_metrics({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "kappa": kap
    })
    
    
    print("Métricas de Regresión Logística:")
    print(f"Precision: {prec:.4f}")
    print(f"Kappa: {kap:.4f}")
    print("-" * 30)
    print(classification_report(y_test, y_pred_lr))

Métricas de Regresión Logística:
Precision: 0.5135
Kappa: 0.3388
------------------------------
              precision    recall  f1-score   support

           0       0.82      0.64      0.72       660
           1       0.51      0.73      0.60       340

    accuracy                           0.67      1000
   macro avg       0.67      0.69      0.66      1000
weighted avg       0.72      0.67      0.68      1000

🏃 View run Logistic_Regression_AndesLink at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/e30e2519f63343648665aa4fc69d402f
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


In [7]:
from sklearn.linear_model import LogisticRegression

# Registro el SEGUNDO experimento de LR en MLflow para comparar
with mlflow.start_run(run_name="Logistic_Regression_V2_Threshold_0.45"):
    
    
    model_lr2 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    
    # Registro de parámetros
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_param("probability_threshold", 0.45) # Registramos el nuevo umbral
    
    # Entrenamiento con tus variables escaladas
    model_lr2.fit(X_train, y_train)
    
    # Uso predict_proba para aplicar el nuevo umbral
    y_probs_lr2 = model_lr2.predict_proba(X_test)[:, 1]
    y_pred_lr2 = (y_probs_lr2 >= 0.45).astype(int)
    
    # Cálculo de métricas con el nuevo umbral
    acc = accuracy_score(y_test, y_pred_lr2)
    prec = precision_score(y_test, y_pred_lr2)
    rec = recall_score(y_test, y_pred_lr2)
    f1 = f1_score(y_test, y_pred_lr2)
    kap = cohen_kappa_score(y_test, y_pred_lr2)
    
    # Registro en MLflow/DagsHub
    mlflow.log_metrics({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "kappa": kap
    })
    
   
    print("Métricas de Regresión Logística V2 (Umbral 0.45):")
    print(f"Recall: {rec:.4f} (Anterior era 0.7294)")
    print(f"Precision: {prec:.4f}")
    print(f"Kappa: {kap:.4f}")
    print("-" * 30)
    print(classification_report(y_test, y_pred_lr2))

Métricas de Regresión Logística V2 (Umbral 0.45):
Recall: 0.7912 (Anterior era 0.7294)
Precision: 0.4963
Kappa: 0.3300
------------------------------
              precision    recall  f1-score   support

           0       0.84      0.59      0.69       660
           1       0.50      0.79      0.61       340

    accuracy                           0.66      1000
   macro avg       0.67      0.69      0.65      1000
weighted avg       0.73      0.66      0.66      1000

🏃 View run Logistic_Regression_V2_Threshold_0.45 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/5feefdbdbd754c919b9b0e0fc7e42406
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


# **Entrenamiento con el Modelo de Random Forest**

In [8]:
# Carga de Experimento al MLflow
with mlflow.start_run(run_name="RandomForest_1"):
    # Connfiguración del modelo con sus parámetros
    n_est1 = 100 
    m_depth1 = 5
    
    rf1_model = RandomForestClassifier(
        n_estimators=n_est1, 
        max_depth=m_depth1, 
        random_state=42,
        class_weight='balanced'
    )
    
    # Registro de parámetros en DagsHub
    mlflow.log_param("n_estimators", n_est1)
    mlflow.log_param("max_depth", m_depth1)
    mlflow.log_param("model_type", "Random Forest")
    
    # Entrenamiento Usando X_train (escalado) y y_train
    rf1_model.fit(X_train, y_train)
    
    # Predicción usando X_test escalado
    y_pred_rf1 = rf1_model.predict(X_test)
    
    # Métricas para el registro en DagsHub  
    # Si en tu split usaste y_test_ml, cambia y_test por y_test_ml abajo
    metrics_rf1 = {
        "accuracy": accuracy_score(y_test, y_pred_rf1),
        "precision": precision_score(y_test, y_pred_rf1),
        "recall": recall_score(y_test, y_pred_rf1),
        "f1_score": f1_score(y_test, y_pred_rf1),
        "kappa": cohen_kappa_score(y_test, y_pred_rf1)
    }
    
    mlflow.log_metrics(metrics_rf1)
    
    # Impresion de reporte detallado para visualización
    print("Métricas RandomForest_1:")
    print(classification_report(y_test, y_pred_rf1))

Métricas RandomForest_1:
              precision    recall  f1-score   support

           0       0.81      0.65      0.72       660
           1       0.51      0.71      0.59       340

    accuracy                           0.67      1000
   macro avg       0.66      0.68      0.66      1000
weighted avg       0.71      0.67      0.68      1000

🏃 View run RandomForest_1 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/da8274799c53460caf459b372ea6ff2e
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


In [9]:
# Carga de Experimento al MLflow
with mlflow.start_run(run_name="RandomForest_2"):
    # Connfiguración del modelo con sus parámetros
    n_est2 = 150
    m_depth2 = 7
    
    rf2_model = RandomForestClassifier(
        n_estimators=n_est2, 
        max_depth=m_depth2, 
        random_state=42,
        class_weight='balanced'
    )
    
    # Registro de parámetros en DagsHub
    mlflow.log_param("n_estimators", n_est2)
    mlflow.log_param("max_depth", m_depth2)
    mlflow.log_param("model_type", "Random Forest")
    
    # Entrenamiento Usando X_train (escalado) y y_train
    rf2_model.fit(X_train, y_train)
    
    # Predicción usando X_test escalado
    y_pred_rf2 = rf2_model.predict(X_test)
    
    # Métricas para el registro en DagsHub  
    # Si en tu split usaste y_test_ml, cambia y_test por y_test_ml abajo
    metrics_rf2 = {
        "accuracy": accuracy_score(y_test, y_pred_rf2),
        "precision": precision_score(y_test, y_pred_rf2),
        "recall": recall_score(y_test, y_pred_rf2),
        "f1_score": f1_score(y_test, y_pred_rf2),
        "kappa": cohen_kappa_score(y_test, y_pred_rf2)
    }
    
    mlflow.log_metrics(metrics_rf2)
    
    # Impresion de reporte detallado para visualización
    print("Métricas RandomForest_2:")
    print(classification_report(y_test, y_pred_rf2))

Métricas RandomForest_2:
              precision    recall  f1-score   support

           0       0.80      0.68      0.73       660
           1       0.52      0.68      0.59       340

    accuracy                           0.68      1000
   macro avg       0.66      0.68      0.66      1000
weighted avg       0.71      0.68      0.68      1000

🏃 View run RandomForest_2 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/90a769a051ac4e888c13bee3810180ab
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


In [10]:
# Carga de Experimento al MLflow
with mlflow.start_run(run_name="RandomForest_3"):
    # Connfiguración del modelo con sus parámetros
    n_est3 = 150
    m_depth3 = 7
    pesos = {0: 1, 1: 1.8}
    
    rf3_model = RandomForestClassifier(
        n_estimators=n_est3, 
        max_depth=m_depth3,
        class_weight=pesos, 
        random_state=42
    )
    
    # Registro de parámetros en DagsHub
    mlflow.log_param("n_estimators", n_est3)
    mlflow.log_param("max_depth", m_depth3)
    mlflow.log_param("model_type", "Random Forest")
    
    # Entrenamiento Usando X_train (escalado) y y_train
    rf3_model.fit(X_train, y_train)
    
    # Predicción usando X_test escalado
    y_pred_rf3 = rf3_model.predict(X_test)
    
    # Métricas para el registro en DagsHub  
    # Si en tu split usaste y_test_ml, cambia y_test por y_test_ml abajo
    metrics_rf3 = {
        "accuracy": accuracy_score(y_test, y_pred_rf3),
        "precision": precision_score(y_test, y_pred_rf3),
        "recall": recall_score(y_test, y_pred_rf3),
        "f1_score": f1_score(y_test, y_pred_rf3),
        "kappa": cohen_kappa_score(y_test, y_pred_rf3)
    }
    
    mlflow.log_metrics(metrics_rf3)
    
    # Impresion de reporte detallado para visualización
    print("Métricas RandomForest_3:")
    print(classification_report(y_test, y_pred_rf3))

Métricas RandomForest_3:
              precision    recall  f1-score   support

           0       0.80      0.72      0.75       660
           1       0.54      0.65      0.59       340

    accuracy                           0.69      1000
   macro avg       0.67      0.68      0.67      1000
weighted avg       0.71      0.69      0.70      1000

🏃 View run RandomForest_3 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/3924dc11397c4087893a9f2a14aab5e2
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


# **Entrenamiento con el Modelo de Naive Bayes**

In [11]:
# Definición de parámetros
umbral1 = 0.50

# Carga de experimento al MLflow
with mlflow.start_run(run_name="NaiveBayes_1"):
    
    nb1_model = GaussianNB()
    
    # Entrenamiento usando X_train escalado y y_train
    nb1_model.fit(X_train, y_train)
    
    # Ajuste de umbral, priorizando Recall para Churn
    y_probs_nb1 = nb1_model.predict_proba(X_test)[:, 1]
    y_pred_nb1 = (y_probs_nb1 >= umbral1).astype(int)
    
    # Cálculo de métricas, usando y_test para comparar
    acc = accuracy_score(y_test, y_pred_nb1)
    rec = recall_score(y_test, y_pred_nb1)
    prec = precision_score(y_test, y_pred_nb1)
    f1 = f1_score(y_test, y_pred_nb1)
    kappa = cohen_kappa_score(y_test, y_pred_nb1)
    
    # Logs para DagsHub
    mlflow.log_param("model_type", "GaussianNB")
    mlflow.log_param("probability_threshold", umbral1)
    
    # Registro de las métricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("kappa", kappa)
    
    # Guardado del modelo en MLflow
    mlflow.sklearn.log_model(nb1_model, "naive_bayes_andeslink1")
    
    # Resultados en consola con formato limpio
    print(f"Métricas NaiveBayes_1 (Umbral {umbral1}):")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Kappa:     {kappa:.4f}")
    
    # Resultado en Consola
    print(f"Métricas NaiveBayes_1 (Umbral {umbral1}):")
    print("-" * 30)
    print(classification_report(y_test, y_pred_nb1)) 
    print(f"Kappa: {kappa:.4f}")

2026/05/09 15:05:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 15:05:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Métricas NaiveBayes_1 (Umbral 0.5):
Accuracy:  0.6720
Recall:    0.6294
Precision: 0.5144
F1 Score:  0.5661
Kappa:     0.3067
Métricas NaiveBayes_1 (Umbral 0.5):
------------------------------
              precision    recall  f1-score   support

           0       0.78      0.69      0.74       660
           1       0.51      0.63      0.57       340

    accuracy                           0.67      1000
   macro avg       0.65      0.66      0.65      1000
weighted avg       0.69      0.67      0.68      1000

Kappa: 0.3067
🏃 View run NaiveBayes_1 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/1f8ab01e072e4d80839385da50a3a71e
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


In [12]:
# Definición de parámetros
umbral2 = 0.35

# Carga de experimento al MLflow
with mlflow.start_run(run_name="NaiveBayes_2"):
    
    nb2_model = GaussianNB()
    
    # Entrenamiento usando X_train escalado y y_train
    nb2_model.fit(X_train, y_train)
    
    # Ajuste de umbral, priorizando Recall para Churn
    y_probs_nb2 = nb2_model.predict_proba(X_test)[:, 1]
    y_pred_nb2 = (y_probs_nb2 >= umbral2).astype(int)
    
    # Cálculo de métricas, usando y_test para comparar
    acc = accuracy_score(y_test, y_pred_nb2)
    rec = recall_score(y_test, y_pred_nb2)
    prec = precision_score(y_test, y_pred_nb2)
    f1 = f1_score(y_test, y_pred_nb2)
    kappa = cohen_kappa_score(y_test, y_pred_nb2)
    
    # Logs para DagsHub
    mlflow.log_param("model_type", "GaussianNB")
    mlflow.log_param("probability_threshold", umbral2)
    
    # Registro de las métricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("kappa", kappa)
    
    # Guardado del modelo en MLflow
    mlflow.sklearn.log_model(nb2_model, "naive_bayes_andeslink2")
    
    # Resultados en consola con formato limpio
    print(f"Métricas NaiveBayes_2 (Umbral {umbral2}):")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Kappa:     {kappa:.4f}")
    
        # Resultado en Consola
    print(f"Métricas NaiveBayes_2 (Umbral {umbral2}):")
    print("-" * 30)
    print(classification_report(y_test, y_pred_nb2)) 
    print(f"Kappa: {kappa:.4f}")

2026/05/09 15:05:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 15:06:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Métricas NaiveBayes_2 (Umbral 0.35):
Accuracy:  0.6360
Recall:    0.7912
Precision: 0.4786
F1 Score:  0.5965
Kappa:     0.2998
Métricas NaiveBayes_2 (Umbral 0.35):
------------------------------
              precision    recall  f1-score   support

           0       0.84      0.56      0.67       660
           1       0.48      0.79      0.60       340

    accuracy                           0.64      1000
   macro avg       0.66      0.67      0.63      1000
weighted avg       0.72      0.64      0.64      1000

Kappa: 0.2998
🏃 View run NaiveBayes_2 at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0/runs/b14fff2b98de46588c0d2d63870f5b75
🧪 View experiment at: https://dagshub.com/JuanManuelResquin84/Proyecto_AndesLink.mlflow/#/experiments/0


# <span style="color:red">**Conclusiones**</span>

### Para un problema de Churn, yo me sigo quedando con el **Modelo Regresión Logistica**, basándome en los siguientes datos:

### Se dio un duelo en los modelos de Regresión Logística y Naive Bayes, ambos modelos son los mejores capturando clientes que se van, con un Recall idéntico de 0.7911. Sin embargo, la de **Regresión Logística con umbral 0.45** gana por calidad de predicción:

### **Mejor Precision: 0.496 vs 0.**478** del Naive Bayes. Esto significa que, al alertar sobre el Churn, la de Regresión Logística se equivoca menos.

### **Mejor F1-Score: Logro 0.609**, que es el valor más alto de los modelos, logrando un equilibrio entre no dejar escapar clientes y no saturar el área de fidelización con falsas alarmas.

### **Mejor Accuracy: 0.656 vs 0.636**, aunque no es la métrica principal, siempre es preferible un modelo que acierte más en el total de los casos.

### Descarte el Modelo de **RandomForest**, aunque tenga el Accuracy más alto (0.693), pero su Recall es muy pobre (0.65). Para AndesLink, esto significa dejar que un 35% de los clientes se vayan sin siquiera detectarlos. Es un modelo demasiado conservador.

### El Modelo **Logistic Regression** con umbral de 0.55 vs 0.45, el primero tienen un buen Accuracy (0.70 vs 0.67), pero su nivel de Recall (0.67 vs 0.72) no alcanzan la meta de detección.

# <span style="color:red">**Conclusión Final**</span>
### El modelo Logistic_Regression_V2_Threshold_0.45 es el más inteligente para el negocio porque:
* Iguala la capacidad de detección máxima (Recall 79%).
* Es más preciso que el Naive Bayes, ahorrando recursos en campañas dirigidas a personas que no pensaban irse.
* Tiene el mejor F1-Score, lo que lo posiciona como el modelo más "maduro" del notebook 02_Modelado_AndesLink.ipynb hasta el momento.

# <span style="color:red">**Empaquetado del Modelo y Escalador para Producción**</span> 
### Al ejecutar estas líneas, estoy asegurandome la reproducibilidad del modelo. No solo guardando el modelo **Logistic_Regression_V2_Threshold_0.45**, sino también el scaler y las columnas, lo cual es indispensable para que el modelo funcione correctamente fuera de tu entorno de entrenamiento.

In [13]:
import joblib
import os

# Defino las rutas de guardado
models_dir = '../models/'

# Defino los nombres de los archivos para la V2 de Regresión Logística
modelo_file = models_dir + 'modelo_churn_lrV2_andeslink.pkl'
scaler_file = models_dir + 'scaler_andeslink.pkl'
columnas_file = models_dir + 'X_columns.pkl'

# Guardo el modelo ganador (model_lr2) y sus complementos
# Uso el modelo entrenado con el umbral de 0.45
joblib.dump(model_lr2, modelo_file) 
joblib.dump(scaler, scaler_file)        # Es el scaler que use en X_train
joblib.dump(X.columns.tolist(), columnas_file) # son las columnas procesadas con dummies

print(f"Modelo exportado")

Modelo exportado


## **Exportando Matrix de Confusion a PNG**

In [14]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

# Configuración de Ruta de guardado
reports_dir = '../reports/'
if not os.path.exists(reports_dir):
    os.makedirs(reports_dir)

#Definición de modelos, títulos y umbrales
configuracion_reporte = [
    {
        "titulo": "1. Logistic Regression (Original - 0.5)",
        "modelo": model_lr, 
        "umbral": 0.5,
        "tipo_pred": "proba" 
    },
    {
        "titulo": "2. Logistic Regression (V2 - 0.45)",
        "modelo": model_lr2, 
        "umbral": 0.45,
        "tipo_pred": "proba"
    },
    {
        "titulo": "3. RandomForest (V1 - D5)",
        "modelo": rf1_model, 
        "umbral": None, 
        "tipo_pred": "direct"
    },
    {
        "titulo": "4. RandomForest (V2 - D7)",
        "modelo": rf2_model, 
        "umbral": None,
        "tipo_pred": "direct"
    },
    {
        "titulo": "5. RandomForest (V3 - D7/Pesos)",
        "modelo": rf3_model, 
        "umbral": None,
        "tipo_pred": "direct"
    },
    {
        "titulo": "6. Naive Bayes (V1 - 0.5)",
        "modelo": nb1_model,
        "umbral": 0.5,
        "tipo_pred": "proba"
    },
    {
        "titulo": "7. Naive Bayes (V2 - 0.35)",
        "modelo": nb2_model, 
        "umbral": 0.35,
        "tipo_pred": "proba"
    }
]

# Configuración del Gráfico 
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
fig.suptitle('Comparativa de Matrices de Confusión - Proyecto AndesLink', fontsize=20, fontweight='bold', color='red')

# Etiquetas para los ejes
labels = ['No Churn (0)', 'Churn (1)']

# Iteración sobre los modelos para generar los gráficos
for i, config in enumerate(configuracion_reporte):
   
    row = i // 4
    col = i % 4
    ax = axes[row, col]
    
    # Genero las predicciones aplicando la lógica de umbral si corresponde
    if config["tipo_pred"] == "proba":
        y_probs = config["modelo"].predict_proba(X_test)[:, 1]
        y_pred = (y_probs >= config["umbral"]).astype(int)
    else:
        y_pred = config["modelo"].predict(X_test)
        
    # Calculo de la matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    
    # Grafico del Heatmap usando Seaborn
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax, cbar=False,
                xticklabels=labels, yticklabels=labels, annot_kws={"size": 14, "weight": "bold"})
    
    # Configuraciones de títulos y etiquetas por subgráfico
    ax.set_title(config["titulo"], fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicción', fontsize=12)
    ax.set_ylabel('Realidad', fontsize=12)

# Limpieza de subgráficos vacíos 
if len(configuracion_reporte) < 8:
    axes[1, 3].axis('off')

# Ajuste automático del diseño para que no se superpongan
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

# Exportación de la Imagen a la carpeta reports
output_image = os.path.join(reports_dir, 'comparativa_matrices_confusion.png')
plt.savefig(output_image, dpi=300, bbox_inches='tight')

# Cierro el gráfico para liberar memoria
plt.close(fig)

print(f"Reporte generado")

Reporte generado


# **Exportando Métricas a PNG**

In [15]:
import mlflow
import pandas as pd
import os
import matplotlib.pyplot as plt

# Configuro la ruta de guardado
reports_dir = '../reports/'
if not os.path.exists(reports_dir):
    os.makedirs(reports_dir)

# Extraigo datos de MLflow
current_experiment = mlflow.get_experiment_by_name("Default")
if current_experiment is None:
    runs = mlflow.search_runs()
else:
    runs = mlflow.search_runs(experiment_ids=[current_experiment.experiment_id])

# Selección y limpieza de columnas
columnas_interes = [
    'tags.mlflow.runName', 
    'params.model_type', 
    'params.probability_threshold',
    'metrics.accuracy', 
    'metrics.recall', 
    'metrics.precision', 
    'metrics.f1_score', 
    'metrics.kappa'
]

columnas_presentes = [c for c in columnas_interes if c in runs.columns]
df_reporte = runs[columnas_presentes].copy()
df_reporte.columns = [c.replace('tags.mlflow.', '').replace('params.', '').replace('metrics.', '') for c in df_reporte.columns]

if 'f1_score' in df_reporte.columns:
    df_reporte = df_reporte.sort_values(by='f1_score', ascending=False)

# Redondeamos a 4 decimales para que la imagen se vea limpia
df_reporte = df_reporte.round(4)


# Defino el tamaño de la imagen según la cantidad de modelos
fig, ax = plt.subplots(figsize=(14, len(df_reporte) * 0.8)) 
ax.axis('off')

# Creo la tabla visual
tabla = ax.table(cellText=df_reporte.values, 
                 colLabels=df_reporte.columns, 
                 cellLoc='center', 
                 loc='center',
                 colColours=["#f2f2f2"] * len(df_reporte.columns)) # Color gris claro para encabezados

# Estilo de la tabla
tabla.auto_set_font_size(False)
tabla.set_fontsize(11)
tabla.scale(2.5, 2.5) # Ajusta el alto de las filas para que no se amontonen

# Título de la imagen
plt.title('Comparativa Final de Métricas - Proyecto AndesLink', fontsize=16, fontweight='bold', pad=20)

# Guardo la imagen en reports
img_path = os.path.join(reports_dir, 'tabla_comparativa_modelos_final.png')
plt.savefig(img_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Tabla exportada")

# Mostrar la tabla en el notebook
df_reporte

Tabla exportada


,runName,model_type,probability_threshold,accuracy,recall,precision,f1_score,kappa
5,Logistic_Regression_V2_Threshold_0.45,Logistic Regression,0.45,0.656,0.7912,0.4963,0.6100,0.3300
6,Logistic_Regression_AndesLink,Logistic Regression,None,0.673,0.7294,0.5135,0.6027,0.3388
0,NaiveBayes_2,GaussianNB,0.35,0.636,0.7912,0.4786,0.5965,0.2998
4,RandomForest_1,Random Forest,None,0.668,0.7059,0.5085,0.5911,0.3239
2,RandomForest_3,Random Forest,None,0.693,0.6500,0.5403,0.5901,0.3480
3,RandomForest_2,Random Forest,None,0.677,0.6765,0.5192,0.5875,0.3295
1,NaiveBayes_1,GaussianNB,0.5,0.672,0.6294,0.5144,0.5661,0.3067


# **Exportando Métricas a JSON**

In [16]:
import mlflow
import json
import os

# Configuración de ruta
reports_dir = '../reports/'
if not os.path.exists(reports_dir):
    os.makedirs(reports_dir)

# Obtener datos de MLflow
runs = mlflow.search_runs()

# Identificar automáticamente todas las métricas y parámetros registrados
# Buscamos columnas que empiecen con 'metrics.' o 'params.' o sean el nombre
cols_metricas = [c for c in runs.columns if c.startswith('metrics.')]
cols_params = [c for c in runs.columns if c.startswith('params.')]
cols_info = ['tags.mlflow.runName'] if 'tags.mlflow.runName' in runs.columns else []

# Combinamos todas las columnas encontradas
todas_las_columnas = cols_info + cols_params + cols_metricas

# Crear el DataFrame y limpiar los nombres
df_final = runs[todas_las_columnas].copy()

# Limpiamos los encabezados (quitamos 'metrics.', 'params.', etc.)
df_final.columns = [c.replace('metrics.', '').replace('params.', '').replace('tags.mlflow.', '') for c in df_final.columns]

# Guardar como archivo .json
json_path = os.path.join(reports_dir, 'metricas_andeslink_completo.json')

# Convertimos a lista de diccionarios
resultado_json = df_final.to_dict(orient='records')

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(resultado_json, f, indent=4, ensure_ascii=False)

print(f"JSON generado")

JSON generado


# **Extracción del Metadato del Modelo Ganador en  JSON**

In [17]:
import json
from datetime import datetime
import os

# Definimos los datos del ganador, Logistic Regression V2

metadata_ganador = {
    "nombre_modelo": "AndesLink_Churn_LR_V2",
    "version": "2.0",
    "autor": "Juan Manuel Resquin",
    "fecha_exportacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "algoritmo": "Logistic Regression",
    "metricas_validacion": {
        "accuracy": 0.656,  
        "recall": 0.791,
        "f1_score": 0.61,
        "umbral_optimo": 0.45
    },
    "variables_entrada": [
        "tenure", "MonthlyCharges", "TotalCharges", 
        "InternetService", "Contract", "PaymentMethod"
    ],
    "entorno_conda": "churn_env"
}

# Guardar el JSON al lado del modelo
models_dir = '../models/'
json_path = os.path.join(models_dir, 'modelo_churn_lrV2_andeslink.json')

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metadata_ganador, f, indent=4, ensure_ascii=False)

print(f"JSON del modelo creado")

JSON del modelo creado
